# 08 Camera model with hand-built features
Notebook 07 showed that a CNN on raw keypoints overfits (macro F1 0.37, near chance) with 4 people. Here each 2 s window becomes ~25 hand-built numbers (`video_pose.pose_features`): how fast the ankles, knees and wrists move, how regular the leg swing is, knee angle, and so on. Same leave-one-person-out split and classes as notebook 07.

In [1]:
import sys, yaml, numpy as np, pandas as pd
sys.path.insert(0, "../src")
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from video_pose import load_video, feature_windows, FEATURE_NAMES
cfg = yaml.safe_load(open("../config/config.yaml")); CL = cfg["video"]["classes"]
d = load_video("../data/video/keypoints", cfg)
F, y, sub = feature_windows(d["X"], cfg["video"]["fps"]), d["y"], d["subject"]
F.shape

(493, 23)

In [2]:
# p5 has walking clips only: used for training, never as test or validation person
MODELS = {"RF": lambda: RandomForestClassifier(300, min_samples_leaf=3, class_weight="balanced", random_state=42, n_jobs=-1),
          "LogReg": lambda: make_pipeline(StandardScaler(), LogisticRegression(C=0.3, class_weight="balanced", max_iter=2000))}
people = sorted(set(sub) - {"p5"}); rows, preds = [], {}
for mn, mk in MODELS.items():
    for t in people:
        te = sub == t; z = mk().fit(F[~te], y[~te]).predict(F[te])
        rows.append({"model": mn, "test person": t, "acc": accuracy_score(y[te], z), "macro F1": f1_score(y[te], z, average="macro")})
        preds.setdefault(mn, []).append((y[te], z))
res = pd.DataFrame(rows)
res.pivot_table(index="test person", columns="model", values="macro F1").round(3).assign()
summ = res.groupby("model")[["acc", "macro F1"]].agg(["mean", "std"]).round(3); summ

acc       macro F1       
         mean   std     mean    std
model                              
LogReg  0.615  0.17    0.609  0.150
RF      0.621  0.15    0.627  0.112

In [3]:
print(res.pivot_table(index="test person", columns="model", values=["acc", "macro F1"]).round(3))
fig, axs = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, (mn, L) in zip(axs[:2], preds.items()):
    yt, yp = np.concatenate([a for a, _ in L]), np.concatenate([b for _, b in L])
    ConfusionMatrixDisplay(confusion_matrix(yt, yp, labels=range(len(CL))), display_labels=CL).plot(ax=ax, colorbar=False)
    ax.set_title(f"{mn} on hand-built features (all 4 folds)")
rf = MODELS["RF"]().fit(F, y); imp = pd.Series(rf.feature_importances_, FEATURE_NAMES).sort_values()[-10:]
imp.plot.barh(ax=axs[2], title="RF top-10 features"); fig.tight_layout()
fig.savefig("../reports/figures/video_features.png", dpi=120)
imp[::-1].round(3)

               acc        macro F1       
model       LogReg     RF   LogReg     RF
test person                              
p1           0.838  0.803    0.820  0.760
p2           0.424  0.455    0.467  0.512
p3           0.597  0.555    0.572  0.560
p4           0.602  0.673    0.576  0.675


ankle_dy_std      0.108
torso_tilt        0.098
knee_ang_min      0.087
spd_ankle         0.085
knee_y_mean       0.072
ankle_dist_std    0.072
knee_dy_std       0.066
knee_ang_range    0.055
knee_ang_mean     0.042
spd_knee          0.041
dtype: float64

Final model for the app's Video page: the Random Forest on hand-built features (best mean macro F1 above), trained on all 5 people.

In [4]:
import joblib
final = MODELS["RF"]().fit(F, y)
joblib.dump({"model": final, "classes": CL, "features": FEATURE_NAMES}, "../models/video_rf.joblib")
print("saved ../models/video_rf.joblib")

saved ../models/video_rf.joblib
